# Task 1: Dataset Selection



!pip install -U datasets transformers seqeval pyarrow

In [2]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "en")

In [4]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


In [5]:
label_list = dataset["train"].features["ner_tags"].feature.names
print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


#  Dataset Used
Dataset: WikiANN (English)

Task: Token Classification (NER / Chunking)

Label Categories
O → Outside

B-PER → Beginning of Person

I-PER → Inside Person

B-ORG → Organization

B-LOC → Location



# TASK 2: Data Preprocessing (Tokenization + Label Alignment)

#   Subword Tokenization
Words may split into multiple tokens

Example: "playing" → ["play", "##ing"]

 Label Alignment
First token → actual label

Other tokens → -100 (ignored)

 Special Tokens
[CLS], [SEP]

Assigned -100



In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [4]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized_inputs.word_ids()

    labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)  # special tokens
        elif word_idx != previous_word_idx:
            labels.append(example["ner_tags"][word_idx])
        else:
            labels.append(-100)  # subword tokens

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [5]:
tokenized_dataset = dataset.map(tokenize_and_align_labels)

In [9]:
print(tokenized_dataset["train"][0])

{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River'], 'input_ids': [101, 155, 119, 145, 119, 16029, 113, 1457, 119, 4898, 1595, 114, 113, 5306, 1604, 13277, 114, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, -100, -100, -100, 4, 0, 3, -100, 4, 4, 0, 0, 0, -100, 0, 0, -100]}


# TASK 3: Model Setup (BERT / DistilBERT)



In [15]:
from transformers import AutoModelForTokenClassification

In [19]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "en")

print("Dataset loaded successfully ✅")

Dataset loaded successfully ✅


In [18]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


In [22]:
label_list = dataset["train"].features["ner_tags"].feature.names
print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [21]:
num_labels = len(label_list)
print("Number of labels:", num_labels)

Number of labels: 7


In [23]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(id2label)
print(label2id)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}


In [24]:
from transformers import AutoModelForTokenClassification

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print("Model loaded ✅")

C:\Users\raniy\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 263.26 MB. The target location C:\Users\raniy\.cache\huggingface\hub\models--distilbert-base-cased\blobs only has 13.45 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

# TASK 4: Training (Using Trainer API)
 Training Details
Learning Rate: 2e-5

Epochs: 3

Batch Size: 16
Trainer API
Simplifies training loop

Handles evaluation & logging




In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

print("Tokenizer loaded ✅")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\raniy\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raniy\.cache\huggingface\hub\models--distilbert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded ✅


In [8]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [4]:
print(tokenizer)

BertTokenizer(name_or_path='distilbert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [5]:
!pip install -U transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [21]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    do_eval=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
trainer.train()

# Task 5: Evaluation
evalution metric:
we use the seqeval library for evaluating token classification perfoemance.
Metrics User


Percision


Recall


F1 Score


Purpose:
these metrics measure how well the model identifies and classifies tokens in sequence labeling tasks, providing a balanced evalution of performance.
Explanation of Metrics
Percision: Measures how many predicted labels are correct out of all predicted labels
Recall: Measures how many actual labels are currectly identified out of all true babels
F1 Score: Harmonic mean of percision and recall , providing a balanced measure of perfoemance
Accuracy: Overall percentage of correctly predicted labels



# Task 6: Inference
Objective:
we test the tarined model on custom input sentences to evaluate its real-world prediction capability.
Process:
Provide a custom input sentance
Tokenize using the same DistilBERT the trained model
pass the tokenized input through the trained model
Decode predicted outputs into label names
Display token-wise predicted labels

In [ ]:
sentence = "Someshwar work at Google in Pune"

# Tokenize properly
inputs = tokenizer(sentence, return_tensors="pt")

# Move input to same device as model
inputs = {k: v.to(device) for k, v in input.items()}

# get predictionsa
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)

# Convert tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Map labels
predicted_labels = [label_list[p.item()] for p in predictions[0]]

# Print output ( ignore special tokens)
for token, label in zip(tokens, predicted_labels):
  if token not in ["[CLS]", "[SEL]"]:
    print(f"{token}: {label}")

# TASK 7: Comparison (POS Tagging vs Chunking)
 Comparison: POS Tagging vs Chunking
Feature	           POS Tagging	                             Chunking
Definition	Assigns grammatical labels to each word	      Groups words into meaningful                                                             phrases
Level	    Word-level	                                  Phrase-level
Example	    Noun, Verb, Adjective	                      Noun Phrase (NP), Verb Phrase                                                            (VP)
Complexity	Easy	                                      Medium
Output	    Tag for each word	                          Group of words (phrases)
Use Case	Grammar analysis	                          Information extraction

Explanation
POS Tagging:

Identifies grammatical role of each word

Example:

John → Noun

works → Verb

Chunking:

Groups words into phrases

Example:

"John works" → Noun Phrase

"at Google" → Prepositional Phrase

Key Difference
POS Tagging → focuses on individual words

Chunking → focuses on groups of words (phrases)

Difficulty Level
POS Tagging → Easy

Chunking → Moderate (requires understanding of structure)

Conclusion
POS tagging is a basic NLP task used for grammatical analysis

Chunking is more advanced and helps in extracting meaningful phrases from text

Both are essential for downstream NLP applications


# TASK 8: Report / Blog

📌 Report: Fine-Tuning Transformer for Token Classification
🔹 Introduction
In this assignment, we implemented a token classification system using a transformer model to perform Named Entity Recognition (NER), which is similar to chunking. We used a pre-trained model and fine-tuned it on a labeled dataset.

🔹 Model Used
We used DistilBERT for this task because it is lightweight, faster, and provides good performance compared to BERT.

🔹 Tasks Performed
Dataset loading and exploration

Tokenization using transformer tokenizer

Label alignment with tokens

Model fine-tuning

Evaluation using precision, recall, and F1 score

Inference on custom sentences

🔹 Difference Between POS Tagging and Chunking
POS Tagging assigns grammatical labels to each word

Chunking groups words into meaningful phrases

POS is simpler, while chunking is more complex

🔹 Challenges Faced
Handling subword tokenization

Aligning labels with split tokens

Managing runtime errors due to missing variables

Training time was high for large models

🔹 Observations
The model successfully learned contextual patterns

Training loss decreased over time

Evaluation metrics showed good performance

DistilBERT provided faster results with decent accuracy
# Conclusion
This project demonstrated how transformer models can be effectively used for sequence labeling tasks like POS tagging and chunking. It also highlighted the importance of preprocessing and correct label alignment in NLP tasks.